# Exercice 1

## Question 1

### 1. Input Space
* Input space ($\mathcal{X}$) : $\mathcal{X} = \mathbb{R}$. Input space is the temperature of and engine bay of a car in Celcius.

### 2. Output Space
* Output space ($\mathcal{Y}$) : $\mathcal{Y} = \{0, 1\}$. Output space is the state of the engine (no unit) :
    - 1 : Broken , overheated
    - 0 : Working normally


### 3. Joint Distribution $P(X, Y)$

* Marginal Distribution of $X$: We suppose that the operational temperature across the fleet of engines follows a normal distribution with a mean $\mu = 90°C$ and a standard deviation $\sigma = 15°C$:
  $$X \sim \mathcal{N}(90, 15^2)$$
* Conditional Distribution of $Y$ given $X=x$: We use a sigmoid function for this problem because the probability of the engine crashing because of the temperature increases when the temperature is higher. We suppose that the threshold is 100°C because a car generally cooldown at around 90-95°C
  $$P(Y=1|X=x) = \frac{1}{1 + e^{-(x-100)}}$$
  $$P(Y=0|X=x) = 1 - P(Y=1|X=x) = \frac{e^{-(x-100)}}{1 + e^{-(x-100)}}$$

### 4. Loss Function
We choose the standard "0-1" loss function, because this is good for classifications problem:
$$l(y, u) = \mathbf{1}_{y \neq u} = \begin{cases} 0 & \text{if } y = u \\ 1 & \text{if } y \neq u \end{cases}$$

### 5. Computation of the Bayes Predictor $f^*(x)$
By definition, the Bayes predictor is the function that minimizes the conditional expected loss (conditional risk) for each observation $x$:
$$f^*(x) = \arg\min_{u \in \{0,1\}} \mathbb{E}[l(Y, u) | X=x]$$

Under the 0-1 loss, the expected loss corresponds exactly to the probability of making a misclassification:
$$\mathbb{E}[l(Y, u) | X=x] = 0 \times P(Y=u|X=x) + 1 \times P(Y \neq u|X=x) = P(Y \neq u|X=x)$$

To minimize this quantity, the optimal strategy is to choose the class with the highest conditional probability:
$$f^*(x) = \begin{cases} 1 & \text{if } P(Y=1|X=x) \ge 0.5 \\ 0 & \text{otherwise} \end{cases}$$

If we solve this inequality:
$$\frac{1}{1 + e^{-(x-100)}} \ge \frac{1}{2} \iff 1 + e^{-(x-100)} \le 2 \iff e^{-(x-100)} \le 1 \iff -(x-100) \le 0 \iff x \ge 100$$

Thus, the Bayes predictor $f^*(x)$ simplifies to a deterministic thresholding function:
$$f^*(x) = \mathbf{1}_{x \ge 100} = \begin{cases} 1 & \text{if } x \ge 100\text{ °C} \\ 0 & \text{otherwise} \end{cases}$$

### 6. Associated Bayes Risk
The Bayes risk $R(f^*)$ represents the absolute lower bound of the generalization error in this setting:
$$R(f*) = \mathbb{E}_X [ \min(P(Y=0|X), P(Y=1|X)) ]$$
$$R(f^*) = \int_{\mathbb{R}} \min\left(1 - \frac{1}{1 + e^{-(x-100)}}, \frac{1}{1 + e^{-(x-100)}}\right) \cdot \frac{1}{15\sqrt{2\pi}} e^{-\frac{(x-90)^2}{2 \times 15^2}} dx$$

Since this integral does not yield a simple closed-form analytical solution, we compute its exact value using a high-precision Monte Carlo numerical approximation in the code cell below.

## Question 2

In [15]:
import numpy as np

# Set random seed to ensure reproducibility of the simulation
np.random.seed(42)

# Define the joint distribution parameters
mu_X, sigma_X = 90, 15
optimal_threshold = 100
n_samples = 250000

# Sample the test set from the true joint distribution P(X, Y)
X_test = np.random.normal(mu_X, sigma_X, n_samples)
conditional_prob = 1 / (1 + np.exp(-(X_test - optimal_threshold)))
# Bernoulli trial to generate the true ground-truth labels Y
Y_test = np.random.binomial(1, conditional_prob)

# Define the predictors
def f_optimal(x):
    """ Theoretical Bayes Predictor (Optimal threshold at 100°C) """
    return (x >= optimal_threshold).astype(int)

def f_suboptimal(x):
    """ Proposed alternative estimator tilde{f} (Suboptimal threshold chosen at 105°C) """
    return (x >= 105).astype(int)

#Generate predictions on the test sampel
Y_pred_bayes = f_optimal(X_test)
Y_pred_tilde = f_suboptimal(X_test)

# Compute empirical risks
empirical_risk_bayes = np.mean(Y_test != Y_pred_bayes)
empirical_risk_tilde = np.mean(Y_test != Y_pred_tilde)

# Compute the Bayes Risk with a dense Monte Carlo approximation
X_large = np.random.normal(mu_X, sigma_X, 1000000)  #to reduce variance
p_Y1 = 1 / (1 + np.exp(-(X_large - optimal_threshold)))
theoretical_bayes_risk = np.mean(np.minimum(p_Y1, 1 - p_Y1))


print(f"Theoretical Bayes Risk (MC)     : {theoretical_bayes_risk:.5f}")
print(f"Empirical Risk of Bayes Predictor f* : {empirical_risk_bayes:.5f}")
print(f"Empirical Risk of Alternative Estimator f~ : {empirical_risk_tilde:.5f}")

print(f"Verification 1: R(f*) < R(f~)               ->  {empirical_risk_bayes < empirical_risk_tilde}")
print(f"Verification 2: Error gap |R_n(f*) - R(f*)|  ->  {abs(empirical_risk_bayes - theoretical_bayes_risk):.5f}")


Theoretical Bayes Risk (MC)     : 0.02927
Empirical Risk of Bayes Predictor f* : 0.02943
Empirical Risk of Alternative Estimator f~ : 0.09586
Verification 1: R(f*) < R(f~)               ->  True
Verification 2: Error gap |R_n(f*) - R(f*)|  ->  0.00016
